In [ ]:
!pip -q install earthengine-api geemap

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries installed and imported successfully.")


In [ ]:
# Google Colab storage configuration
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Change this folder if your project data are stored elsewhere in Google Drive.
DATA_DIR = Path('/content/drive/MyDrive/FloodSense_AI')
DATA_DIR.mkdir(parents=True, exist_ok=True)

UNOSAT_ZIP = DATA_DIR / 'UNOSAT_Flood.zip'
UNOSAT_DIR = DATA_DIR / 'UNOSAT_Reference'

print('Project data directory:', DATA_DIR)


In [ ]:
ee.Authenticate()

# Replace with a Google Cloud project ID that has Earth Engine enabled.
EE_PROJECT = "YOUR_GOOGLE_CLOUD_PROJECT_ID"
if EE_PROJECT == "YOUR_GOOGLE_CLOUD_PROJECT_ID":
    raise ValueError("Set EE_PROJECT to your own Google Cloud project ID before continuing.")
ee.Initialize(project=EE_PROJECT)

print("Google Earth Engine connected successfully.")


In [ ]:

# Load administrative boundaries
gaul = ee.FeatureCollection('FAO/GAUL/2015/level2')

# Select administrative boundaries in Pakistan
pakistan = gaul.filter(
    ee.Filter.eq('ADM0_NAME', 'Pakistan')
)

# Search for Nowshera District
nowshera = pakistan.filter(
    ee.Filter.stringContains('ADM2_NAME', 'Nowshera')
)

# Display the search results
print("Matching features:", nowshera.size().getInfo())

print(
    "District names:",
    nowshera.aggregate_array('ADM2_NAME').getInfo()
)


In [ ]:

# Extract the district geometry
roi = nowshera.geometry()

# Calculate district area in square kilometres
area_km2 = roi.area().divide(1e6)

# Display the area
print(
    "Nowshera District area:",
    round(area_km2.getInfo(), 2),
    "km²"
)

# Confirm geometry preparation
print("Study area prepared successfully!")


In [ ]:
# Load Sentinel-1 satellite data
sentinel1 = ee.ImageCollection('COPERNICUS/S1_GRD')

# Filter images covering Nowshera during 2025
s1_2025 = (
    sentinel1
    .filterBounds(roi)
    .filterDate('2025-01-01', '2026-01-01')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(
        ee.Filter.listContains(
            'transmitterReceiverPolarisation', 'VV'
        )
    )
    .filter(ee.Filter.eq('resolution_meters', 10))
)

# Count available images
image_count = s1_2025.size().getInfo()

print("Sentinel-1 images available in 2025:", image_count)


In [ ]:
# Extract acquisition dates from Sentinel-1 images
dates = s1_2025.aggregate_array('system:time_start').getInfo()

# Convert timestamps into readable dates
from datetime import datetime, timezone

acquisition_dates = sorted(set(
    datetime.fromtimestamp(
        date / 1000, timezone.utc
    ).strftime('%Y-%m-%d')
    for date in dates
))

# Display acquisition dates
print("Total unique acquisition dates:", len(acquisition_dates))

print("\nAvailable Sentinel-1 dates:")

for date in acquisition_dates:
    print(date)


In [ ]:
# Select Sentinel-1 images around the flood event
s1_aug = s1_2025.filterDate(
    '2025-08-01', '2025-08-22'
)

# Extract important information from each image
def extract_metadata(image):
    return ee.Feature(None, {
        'date': ee.Date(
            image.get('system:time_start')
        ).format('YYYY-MM-dd'),

        'pass': image.get('orbitProperties_pass'),

        'relative_orbit': image.get(
            'relativeOrbitNumber_start'
        )
    })

# Convert image metadata into a FeatureCollection
metadata = s1_aug.map(extract_metadata).getInfo()

# Display the results
print("Images in selected period:", s1_aug.size().getInfo())
print("\nDate | Pass direction | Relative orbit")

for feature in metadata['features']:
    properties = feature['properties']

    print(
        properties['date'],
        "|", properties['pass'],
        "|", properties['relative_orbit']
    )


In [ ]:
# Orbit 173: Ascending
pre_173 = s1_2025.filterDate(
    '2025-08-08', '2025-08-09'
).filter(
    ee.Filter.eq('relativeOrbitNumber_start', 173)
)

post_173 = s1_2025.filterDate(
    '2025-08-20', '2025-08-21'
).filter(
    ee.Filter.eq('relativeOrbitNumber_start', 173)
)

# Orbit 5: Descending
pre_5 = s1_2025.filterDate(
    '2025-08-09', '2025-08-10'
).filter(
    ee.Filter.eq('relativeOrbitNumber_start', 5)
)

post_5 = s1_2025.filterDate(
    '2025-08-21', '2025-08-22'
).filter(
    ee.Filter.eq('relativeOrbitNumber_start', 5)
)

# Print image counts
print("Orbit 173 - Pre-flood:", pre_173.size().getInfo())
print("Orbit 173 - Post-flood:", post_173.size().getInfo())

print("Orbit 5 - Pre-flood:", pre_5.size().getInfo())
print("Orbit 5 - Post-flood:", post_5.size().getInfo())


In [ ]:
# Create median composites for each orbit

# Ascending orbit 173
pre_173_img = pre_173.median().clip(roi)
post_173_img = post_173.median().clip(roi)

# Descending orbit 5
pre_5_img = pre_5.median().clip(roi)
post_5_img = post_5.median().clip(roi)

# Display confirmation
print("Four Sentinel-1 composites created successfully!")


In [ ]:
# Calculate valid radar coverage for an image
def check_coverage(image):

    # Keep pixels where VV radar data is valid
    valid_pixels = ee.Image.pixelArea().updateMask(
        image.select('VV').mask()
    )

    # Calculate valid area in square metres
    valid_area = valid_pixels.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=20,
        maxPixels=1e10,
        tileScale=4
    ).get('area')

    # Calculate total district area
    total_area = roi.area()

    # Calculate percentage of valid coverage
    percentage = ee.Number(valid_area).divide(
        total_area
    ).multiply(100)

    return percentage.getInfo()


# Check all four composites
print("Valid Sentinel-1 coverage:")

print("Orbit 173 pre-flood:",
      round(check_coverage(pre_173_img), 2), "%")

print("Orbit 173 post-flood:",
      round(check_coverage(post_173_img), 2), "%")

print("Orbit 5 pre-flood:",
      round(check_coverage(pre_5_img), 2), "%")

print("Orbit 5 post-flood:",
      round(check_coverage(post_5_img), 2), "%")


In [ ]:
# Create an interactive map
Map = geemap.Map()

# Use satellite imagery as the background
Map.add_basemap('SATELLITE')

# Centre the map on Nowshera
Map.centerObject(roi, 9)

# Radar visualisation settings
vis = {
    'min': -25,
    'max': -5,
    'palette': ['black', 'blue', 'white']
}

# Add Sentinel-1 composites
Map.addLayer(
    pre_173_img.select('VV'),
    vis,
    'Orbit 173 - Pre-flood'
)

Map.addLayer(
    post_173_img.select('VV'),
    vis,
    'Orbit 173 - Post-flood'
)

Map.addLayer(
    pre_5_img.select('VV'),
    vis,
    'Orbit 5 - Pre-flood'
)

Map.addLayer(
    post_5_img.select('VV'),
    vis,
    'Orbit 5 - Post-flood'
)

# Add Nowshera boundary
boundary = nowshera.style(
    color='yellow',
    fillColor='00000000',
    width=2
)

Map.addLayer(boundary, {}, 'Nowshera Boundary')

# Add layer controls
Map.addLayerControl()

# Display map
Map


In [ ]:
# Calculate backscatter change for each orbit

# Ascending orbit 173
diff_173 = post_173_img.select('VV').subtract(
    pre_173_img.select('VV')
).rename('VV_Change')

# Descending orbit 5
diff_5 = post_5_img.select('VV').subtract(
    pre_5_img.select('VV')
).rename('VV_Change')

# Display confirmation
print("Backscatter change calculated successfully!")

print("Orbit 173:", diff_173.bandNames().getInfo())
print("Orbit 5:", diff_5.bandNames().getInfo())


In [ ]:
# Create a new map
ChangeMap = geemap.Map()

# Use satellite background
ChangeMap.add_basemap('SATELLITE')

# Centre map on Nowshera
ChangeMap.centerObject(roi, 9)

# Visualisation settings for backscatter change
change_vis = {
    'min': -5,
    'max': 5,
    'palette': ['blue', 'white', 'red']
}

# Display ascending orbit change
ChangeMap.addLayer(
    diff_173,
    change_vis,
    'Ascending - Orbit 173',
    True
)

# Add district boundary
ChangeMap.addLayer(
    nowshera.style(
        color='yellow',
        fillColor='00000000',
        width=2
    ),
    {},
    'Nowshera Boundary'
)

# Add layer controls
ChangeMap.addLayerControl()

# Display map
ChangeMap


In [ ]:
# Check the numerical values in descending orbit 5
stats_5 = diff_5.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=roi,
    scale=30,
    maxPixels=1e10,
    tileScale=4
)

# Display the results
print("Descending orbit 5 statistics:")
print(stats_5.getInfo())


In [ ]:
# Create a separate map for descending orbit
DescendingMap = geemap.Map()

# Add satellite background
DescendingMap.add_basemap('SATELLITE')

# Centre on Nowshera
DescendingMap.centerObject(roi, 9)

# Set a wider display range
descending_vis = {
    'min': -10,
    'max': 10,
    'palette': ['blue', 'white', 'red']
}

# Display only descending orbit change
DescendingMap.addLayer(
    diff_5,
    descending_vis,
    'Descending Orbit 5 Backscatter Change'
)

# Add district boundary
DescendingMap.addLayer(
    nowshera.style(
        color='yellow',
        fillColor='00000000',
        width=2
    ),
    {},
    'Nowshera Boundary'
)

# Display the map
DescendingMap


In [ ]:
# Function to calculate backscatter statistics
def calculate_stats(image):

    stats = image.reduceRegion(
        reducer=ee.Reducer.minMax().combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.median(),
            sharedInputs=True
        ),
        geometry=roi,
        scale=30,
        maxPixels=1e10,
        tileScale=4
    )

    return stats.getInfo()


# Calculate statistics
ascending_stats = calculate_stats(diff_173)
descending_stats = calculate_stats(diff_5)

# Display results
print("ASCENDING ORBIT 173")
print(ascending_stats)

print("\nDESCENDING ORBIT 5")
print(descending_stats)


In [ ]:
print("Ascending image bands:", diff_173.bandNames().getInfo())
print("Descending image bands:", diff_5.bandNames().getInfo())

In [ ]:
import matplotlib.pyplot as plt

ascending_values = diff_173.reduceRegion(
    reducer=ee.Reducer.histogram(
        maxBuckets=50
    ),
    geometry=roi,
    scale=30,
    maxPixels=1e10,
    tileScale=4
).getInfo()

histogram = ascending_values['VV_Change']

plt.figure(figsize=(9, 5))
plt.bar(
    histogram['bucketMeans'],
    histogram['histogram'],
    width=0.5
)

plt.xlabel('VV Backscatter Change (dB)')
plt.ylabel('Number of Pixels')
plt.title('Ascending Orbit 173: Backscatter Change Distribution')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
descending_values = diff_5.reduceRegion(
    reducer=ee.Reducer.histogram(maxBuckets=50),
    geometry=roi,
    scale=30,
    maxPixels=1e10,
    tileScale=4
).getInfo()

histogram = descending_values['VV_Change']

plt.figure(figsize=(9, 5))
plt.bar(
    histogram['bucketMeans'],
    histogram['histogram'],
    width=0.5
)

plt.xlabel('VV Backscatter Change (dB)')
plt.ylabel('Number of Pixels')
plt.title('Descending Orbit 5: Backscatter Change Distribution')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Define three negative-change thresholds
thresholds = [-3, -5, -8]

# Calculate the percentage of pixels below each threshold
change_flags = ee.Image.cat([
    diff_173.lt(t).rename(f'Ascending_below_{abs(t)}dB')
    for t in thresholds
] + [
    diff_5.lt(t).rename(f'Descending_below_{abs(t)}dB')
    for t in thresholds
])

# Calculate the mean of the binary images
percentages = change_flags.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=30,
    maxPixels=1e10,
    tileScale=4
).getInfo()

# Convert proportions into percentages
for band, value in percentages.items():
    print(f'{band}: {value * 100:.2f}%')

In [ ]:
# Create a common valid-data mask
common_mask = (
    diff_173.mask().And(diff_5.mask())
)

# Identify pixels with changes below -5 dB
ascending_flood_candidates = diff_173.lt(-5)
descending_flood_candidates = diff_5.lt(-5)

# Calculate areas using the same valid-data footprint
pixel_area = ee.Image.pixelArea()

ascending_area = pixel_area.updateMask(
    ascending_flood_candidates.And(common_mask)
)

descending_area = pixel_area.updateMask(
    descending_flood_candidates.And(common_mask)
)

overlap_area = pixel_area.updateMask(
    ascending_flood_candidates.And(
        descending_flood_candidates
    ).And(common_mask)
)

union_area = pixel_area.updateMask(
    ascending_flood_candidates.Or(
        descending_flood_candidates
    ).And(common_mask)
)

# Calculate all areas in one operation
area_results = ee.Image.cat([
    ascending_area.rename('Ascending_km2'),
    descending_area.rename('Descending_km2'),
    overlap_area.rename('Overlap_km2'),
    union_area.rename('Union_km2')
]).divide(1e6).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=30,
    maxPixels=1e10,
    tileScale=4
).getInfo()

print("Candidate negative-change areas:")
for name, value in area_results.items():
    print(f"{name}: {value:.2f} km²")

# Calculate spatial agreement using the Jaccard index
jaccard = (
    area_results['Overlap_km2'] /
    area_results['Union_km2']
)

print(f"\nOrbit agreement (Jaccard Index): {jaccard:.3f}")

In [ ]:
# Identify the four spatial categories
both_orbits = (
    ascending_flood_candidates
    .And(descending_flood_candidates)
)

ascending_only = (
    ascending_flood_candidates
    .And(descending_flood_candidates.Not())
)

descending_only = (
    descending_flood_candidates
    .And(ascending_flood_candidates.Not())
)

# Create a categorical agreement image
agreement_map = (
    ee.Image.constant(0)
    .where(ascending_only, 1)
    .where(descending_only, 2)
    .where(both_orbits, 3)
    .rename('Agreement')
    .updateMask(common_mask)
    .clip(roi)
)

# Visualize the agreement map
AgreementMap = geemap.Map()
AgreementMap.add_basemap('SATELLITE')
AgreementMap.centerObject(roi, 9)

agreement_vis = {
    'min': 0,
    'max': 3,
    'palette': [
        '808080',  # Neither
        'FFA500',  # Ascending only
        '0000FF',  # Descending only
        '00FF00'   # Both
    ]
}

AgreementMap.addLayer(
    agreement_map,
    agreement_vis,
    'Sentinel-1 Orbit Agreement'
)

AgreementMap.addLayer(
    nowshera.style(
        color='yellow',
        fillColor='00000000',
        width=2
    ),
    {},
    'Nowshera Boundary'
)

AgreementMap.addLayerControl()
AgreementMap

In [ ]:
# Load the SRTM elevation model
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)

# Calculate terrain slope in degrees
slope = ee.Terrain.slope(dem)

# Combine elevation, slope and agreement categories
terrain_image = ee.Image.cat([
    dem.rename('Elevation_m'),
    slope.rename('Slope_deg'),
    agreement_map
])

# Calculate mean elevation and slope for each category
for category, label in [
    (0, 'Neither orbit'),
    (1, 'Ascending only'),
    (2, 'Descending only'),
    (3, 'Both orbits')
]:

    result = terrain_image.updateMask(
        agreement_map.eq(category)
    ).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,
        maxPixels=1e10,
        tileScale=4
    ).getInfo()

    print(f'\n{label}')
    print('Mean elevation:', round(result['Elevation_m'], 2), 'm')
    print('Mean slope:', round(result['Slope_deg'], 2), 'degrees')

In [ ]:
# Load historical surface-water occurrence
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')

# Select water occurrence band
water_occurrence = jrc.select('occurrence')

# Identify historically frequent water pixels
permanent_water = water_occurrence.gte(90)

# Exclude historical water from our agreement map
agreement_no_water = agreement_map.updateMask(
    permanent_water.Not()
)

print("Historical water exclusion prepared successfully!")

In [ ]:
# Remove historically frequent water from each candidate category
ascending_final = ascending_only.And(
    permanent_water.Not()
)

descending_final = descending_only.And(
    permanent_water.Not()
)

both_final = both_orbits.And(
    permanent_water.Not()
)

# Create final categorical candidate map
final_flood_map = (
    ee.Image.constant(0)
    .where(ascending_final, 1)
    .where(descending_final, 2)
    .where(both_final, 3)
    .rename('Flood_Candidate')
    .updateMask(common_mask)
    .updateMask(permanent_water.Not())
    .clip(roi)
)

# Display the final map
FinalMap = geemap.Map()
FinalMap.add_basemap('SATELLITE')
FinalMap.centerObject(roi, 9)

final_vis = {
    'min': 0,
    'max': 3,
    'palette': [
        '808080',
        'FFA500',
        '0000FF',
        '00FF00'
    ]
}

FinalMap.addLayer(
    final_flood_map,
    final_vis,
    'Final Flood Candidate Map'
)

FinalMap.addLayer(
    nowshera.style(
        color='yellow',
        fillColor='00000000',
        width=2
    ),
    {},
    'Nowshera District'
)

FinalMap.addLayerControl()
FinalMap

In [ ]:
# Calculate area of each final candidate category
final_area_image = ee.Image.cat([
    ee.Image.pixelArea().updateMask(
        ascending_final
    ).rename('Ascending_only_km2'),

    ee.Image.pixelArea().updateMask(
        descending_final
    ).rename('Descending_only_km2'),

    ee.Image.pixelArea().updateMask(
        both_final
    ).rename('Both_orbits_km2')
]).divide(1e6)

final_areas = final_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=30,
    maxPixels=1e10,
    tileScale=4
).getInfo()

for category, area in final_areas.items():
    print(f'{category}: {area:.2f} km²')

In [ ]:
import zipfile
from pathlib import Path
import os

zip_path = UNOSAT_ZIP
extract_folder = UNOSAT_DIR
extract_folder.mkdir(parents=True, exist_ok=True)

# Guard against path traversal entries in externally supplied ZIP archives.
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    root = extract_folder.resolve()
    for member in zip_ref.infolist():
        target = (extract_folder / member.filename).resolve()
        if os.path.commonpath([str(root), str(target)]) != str(root):
            raise ValueError(f"Unsafe path in ZIP archive: {member.filename}")
    zip_ref.extractall(extract_folder)

print("Files extracted:")
print(os.listdir(extract_folder))


In [ ]:

import os

folder = str(UNOSAT_DIR)

for root, dirs, files in os.walk(folder):
    for file in files:
        if file.endswith(".shp"):
            print(os.path.join(root, file))


In [ ]:
!pip -q install geopandas

import geopandas as gpd

reference_path = str(UNOSAT_DIR / "noshera_flood" / "S2_20250820_FloodExtent_PK.shp")

unosat = gpd.read_file(reference_path)

print("Number of flood features:", len(unosat))
print("Coordinate system:", unosat.crs)
print("Columns:", unosat.columns.tolist())
print("Total reference area (km²):", round(unosat.to_crs(epsg=32643).area.sum() / 1e6, 2))


In [ ]:

# Inspect the information stored in the UNOSAT shapefile

print("Reference attributes:")
print(unosat.drop(columns="geometry").T)

print("\nGeometry type:", unosat.geometry.iloc[0].geom_type)
print("Geographic bounds:", unosat.total_bounds)


In [ ]:

from shapely.geometry import shape
import geopandas as gpd

# Convert our Google Earth Engine district boundary to GeoPandas
district = gpd.GeoDataFrame(
    geometry=[shape(roi.getInfo())],
    crs="EPSG:4326"
)

# Combine UNOSAT flood geometry and intersect with Nowshera
flood_geometry = unosat.geometry.unary_union
intersection = flood_geometry.intersection(district.geometry.iloc[0])

# Calculate areas using an equal-area projection
reference_nowshera = gpd.GeoSeries(
    [intersection], crs="EPSG:4326"
).to_crs(epsg=6933)

district_area = district.to_crs(epsg=6933).area.iloc[0] / 1e6
reference_area = reference_nowshera.area.iloc[0] / 1e6

print("Nowshera district area:", round(district_area, 2), "km²")
print("UNOSAT flood area inside Nowshera:", round(reference_area, 2), "km²")
print("Intersection exists:", not intersection.is_empty)


In [ ]:
print(final_flood_map.bandNames().getInfo())

In [ ]:
import geemap
import ee

# Convert the Shapely intersection into GeoJSON
intersection_geojson = intersection.__geo_interface__

# Convert GeoJSON into an Earth Engine geometry
reference_geometry = ee.Geometry(intersection_geojson)

# Create UNOSAT binary reference image
reference_image = (
    ee.Image.constant(0)
    .byte()
    .paint(
        ee.FeatureCollection([
            ee.Feature(reference_geometry)
        ]),
        1
    )
    .rename('Reference')
    .clip(roi)
)

# Convert Sentinel-1 map into binary form
sentinel_candidate = (
    final_flood_map.gt(0)
    .unmask(0)
    .rename('Sentinel1')
    .clip(roi)
)

# Initialize the Map with a specific open-source SATELLITE basemap
Map = geemap.Map(
    basemap='SATELLITE',
    google_map=False
)

Map.centerObject(roi, 10)

# Show district boundary
Map.addLayer(
    ee.Image().paint(nowshera, 2, 2),
    {'palette': ['FFFFFF']},
    'Nowshera Boundary'
)

# UNOSAT reference
Map.addLayer(
    reference_image.selfMask(),
    {'palette': ['00FFFF']},
    'UNOSAT Reference Flood'
)

# Sentinel-1 candidates
Map.addLayer(
    sentinel_candidate.selfMask(),
    {'palette': ['FF0000']},
    'Sentinel-1 Flood Candidates'
)

Map

In [ ]:
import geemap
import ee

# Initialize a standard map instance
Map = geemap.Map(add_google_map=False)

# Remove all existing basemap/background layers to leave a plain background
for layer in list(Map.layers):
    Map.remove_layer(layer)

# Set map view
Map.centerObject(roi, 10)

# Add Nowshera district boundary in black
boundary_layer = ee.Image().byte().paint(
    nowshera, 1, 2
)

Map.addLayer(
    boundary_layer,
    {'palette': ['000000']},
    'Nowshera Boundary'
)

# Add UNOSAT reference flood in Cyan
Map.addLayer(
    reference_image.selfMask(),
    {'palette': ['00FFFF']},
    'UNOSAT Reference Flood'
)

# Add Sentinel-1 candidates in Red
Map.addLayer(
    sentinel_candidate.selfMask(),
    {'palette': ['FF0000']},
    'Sentinel-1 Flood Candidates'
)

Map

In [ ]:
import ee

# The major river flowing through Nowshera is the Kabul River.
# We can extract the permanent water body (historical water occurrence > 90) which outlines the main river channel.
river_channel = permanent_water.updateMask(permanent_water)

# Let's calculate the distance of both Sentinel-1 candidates and UNOSAT flood pixels to the river channel
# This helps verify if they are located close to the river.
distance_to_river = river_channel.fastDistanceTransform().sqrt().multiply(30) # Distance in meters (approx. 30m scale)

# Mask distance map with flood layer locations
s1_distances = distance_to_river.updateMask(sentinel_candidate.gt(0))
unosat_distances = distance_to_river.updateMask(reference_image.gt(0))

# Reduce regions to get mean distances to the river channel
mean_dist_s1 = s1_distances.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=30,
    maxPixels=1e10
).get('distance')

mean_dist_unosat = unosat_distances.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=30,
    maxPixels=1e10
).get('distance')

print("Average distance of Sentinel-1 Flood Candidates to River Channel:", round(mean_dist_s1.getInfo() or 0, 2), "meters")
print("Average distance of UNOSAT Flood Reference to River Channel:", round(mean_dist_unosat.getInfo() or 0, 2), "meters")


In [ ]:
# Let's find the coordinate center points of the flooded areas for spatial reference
# We convert the geometries of the masked outputs to get their spatial centroids
def get_centroid_lon_lat(binary_image, label_name):
    centroids = binary_image.selfMask().reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,
        maxPixels=1e10
    )
    # Get coordinates of the bounding box to describe location
    proj = binary_image.projection()
    latlon = ee.Image.pixelLonLat().updateMask(binary_image)
    means = latlon.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,
        maxPixels=1e10
    )
    print(f"{label_name} centroid coordinates: Longitude {round(means.get('longitude').getInfo() or 0, 4)}°, Latitude {round(means.get('latitude').getInfo() or 0, 4)}°")

get_centroid_lon_lat(reference_image, "UNOSAT Flood Reference (Cyan)")
get_centroid_lon_lat(sentinel_candidate, "Sentinel-1 Flood Candidates (Red)")


In [ ]:

# 1. Create binary flood layers
reference_binary = reference_image.unmask(0).gt(0)
candidate_binary = final_flood_map.gt(0).unmask(0)

# 2. Use only pixels valid in both Sentinel-1 orbits,
# excluding historically frequent water
evaluation_mask = common_mask.And(permanent_water.Not())

# 3. Encode comparison categories
# 0 = True Negative
# 1 = False Positive
# 2 = False Negative
# 3 = True Positive

comparison = (
    reference_binary.multiply(2)
    .add(candidate_binary)
    .rename('comparison')
    .updateMask(evaluation_mask)
)

# 4. Count pixels at a common 20 m resolution
histogram = comparison.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=roi,
    crs='EPSG:32643',
    scale=20,
    maxPixels=1e10,
    tileScale=4
).get('comparison').getInfo()

print("Confusion matrix pixel counts:", histogram)

# 5. Extract the four categories
TN = histogram.get('0', 0)
FP = histogram.get('1', 0)
FN = histogram.get('2', 0)
TP = histogram.get('3', 0)

# 6. Calculate metrics
precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN) if TP + FN else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)
iou = TP / (TP + FP + FN) if TP + FP + FN else 0

print("\nVALIDATION RESULTS")
print("True Positives:", TP)
print("False Positives:", FP)
print("False Negatives:", FN)
print("True Negatives:", TN)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))
print("IoU:", round(iou, 3))


In [ ]:
# Find the 4 August and 16 August
# Sentinel-1 observations from descending orbit 107

pre_107 = (
    s1_2025
    .filterDate('2025-08-04', '2025-08-05')
    .filter(ee.Filter.eq('relativeOrbitNumber_start', 107))
)

event_107 = (
    s1_2025
    .filterDate('2025-08-16', '2025-08-17')
    .filter(ee.Filter.eq('relativeOrbitNumber_start', 107))
)

print("4 August images:", pre_107.size().getInfo())
print("16 August images:", event_107.size().getInfo())


In [ ]:

# Create pre-event and event-period composites
pre_107_img = pre_107.median().clip(roi)
event_107_img = event_107.median().clip(roi)

# Calculate VV backscatter change
# Negative values indicate a reduction in radar backscatter
diff_107 = (
    event_107_img.select('VV')
    .subtract(pre_107_img.select('VV'))
    .rename('VV_Change_107')
)

# Calculate basic statistics
stats_107 = diff_107.reduceRegion(
    reducer=ee.Reducer.mean()
    .combine(ee.Reducer.median(), sharedInputs=True)
    .combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=roi,
    scale=20,
    maxPixels=1e10
)

print("Orbit 107 change statistics:")
print(stats_107.getInfo())


In [ ]:

# Identify strong negative backscatter changes
candidate_107 = diff_107.lt(-5)

# Exclude historically frequent water
candidate_107_final = (
    candidate_107
    .And(permanent_water.Not())
)

# Calculate candidate area
area_107 = (
    ee.Image.pixelArea()
    .updateMask(candidate_107_final)
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=20,
        maxPixels=1e10
    )
    .get('area')
)

print(
    "Orbit 107 candidate change area:",
    round(ee.Number(area_107).divide(1e6).getInfo(), 2),
    "km²"
)


In [ ]:
# Load UNOSAT flood extent for 16 August
reference_16_path = str(
    UNOSAT_DIR / "noshera_flood" / "S2_20250820_FloodExtent_PK.shp"
)

unosat_16 = gpd.read_file(reference_16_path)

print("Features:", len(unosat_16))
print("Sensor date:", unosat_16['Sensor_Dat'].tolist())
print("Water classification:", unosat_16['Water_Clas'].tolist())
print("Confidence:", unosat_16['Confidence'].tolist())

# Calculate intersection with Nowshera
flood_16_geometry = unosat_16.geometry.unary_union
intersection_16 = flood_16_geometry.intersection(
    district.geometry.iloc[0]
)

area_16 = gpd.GeoSeries(
    [intersection_16], crs="EPSG:4326"
).to_crs(epsg=6933).area.iloc[0] / 1e6

print("UNOSAT 16 August flood area inside Nowshera:",
      round(area_16, 2), "km²")


In [ ]:
# Check existing project data
check_names = [
    "final_flood_map",
    "roi",
    "nowshera",
    "dem",
    "slope",
    "rainfall",
    "ndvi",
    "lulc",
    "elevation",
    "aspect"
]

for name in check_names:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}: available")

        try:
            print("Bands:", obj.bandNames().getInfo())
        except:
            print("Available, but not an Earth Engine image.")
    else:
        print(f"\n{name}: not available")

In [ ]:
# Combine all predictor layers
predictors = (
    dem.rename("elevation")
    .addBands(slope.rename("slope"))
    .addBands(ndvi.rename("ndvi"))
    .addBands(diff_173.rename("ascending_change"))
    .addBands(diff_5.rename("descending_change"))
)

# Check the final predictor bands
print("Predictor bands:")
print(predictors.bandNames().getInfo())

print("\nReference bands:")
print(reference_image.bandNames().getInfo())

In [ ]:

# Combine predictors and reference labels
training_image = predictors.addBands(
    reference_image.rename("flood")
)

# Sample pixels across Nowshera
samples = training_image.stratifiedSample(
    numPoints=3000,
    classBand="flood",
    classValues=[0, 1],
    classPoints=[1500, 1500],
    region=roi,
    scale=20,
    seed=42,
    geometries=True,
    tileScale=4
)

# Bring samples into Python
df = geemap.ee_to_df(samples)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["flood"].value_counts())
print("\nFirst five rows:")
display(df.head())


In [ ]:
# Add geographic coordinates
coordinate_bands = ee.Image.pixelLonLat()

training_image_spatial = (
    predictors
    .addBands(reference_image.rename("flood"))
    .addBands(coordinate_bands)
)

# Recreate the balanced sample with coordinates
samples_spatial = training_image_spatial.stratifiedSample(
    numPoints=3000,
    classBand="flood",
    classValues=[0, 1],
    classPoints=[1500, 1500],
    region=roi,
    scale=20,
    seed=42,
    geometries=False,
    tileScale=4
)

df = geemap.ee_to_df(samples_spatial)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Predictor variables
features = [
    "elevation",
    "slope",
    "ndvi",
    "ascending_change",
    "descending_change"
]

X = df[features]
y = df["flood"]

# Create approximately 5 km spatial blocks
lat_block = np.floor(df["latitude"] / 0.05)
lon_block = np.floor(df["longitude"] / 0.05)

groups = lat_block.astype(str) + "_" + lon_block.astype(str)

# Split entire geographic blocks between training and testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining class distribution:")
print(y_train.value_counts())
print("\nTesting class distribution:")
print(y_test.value_counts())

In [ ]:

!pip -q install xgboost

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

# Account for the training class imbalance
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

# Train XGBoost
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=negative / positive,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

# Predict on unseen geographic blocks
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

# Evaluate model performance
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", round(roc_auc_score(y_test, y_probability), 3))


In [ ]:
!pip -q install shap

import shap
import matplotlib.pyplot as plt

# Explain predictions using a sample of test data
X_explain = X_test.sample(
    n=min(300, len(X_test)),
    random_state=42
)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_explain)

# Show feature importance and direction of influence
shap.summary_plot(
    shap_values,
    X_explain,
    plot_type="dot",
    show=True
)


In [ ]:

# Train Random Forest with class-specific probability output
rf_classifier = (
    ee.Classifier.smileRandomForest(numberOfTrees=100)
    .setOutputMode("MULTIPROBABILITY")
    .train(
        features=samples_spatial,
        classProperty="flood",
        inputProperties=features
    )
)

# Predict probabilities for both classes
probability_array = predictors.select(features).classify(
    rf_classifier
)

# Extract class 1: flood probability
probability_image = (
    probability_array
    .arrayGet([1])
    .rename("flood_probability")
    .clip(roi)
)

# Display the flood probability map
Map.add_layer(
    probability_image,
    {
        "min": 0,
        "max": 1,
        "palette": ["white", "yellow", "orange", "red"]
    },
    "Random Forest Flood Probability"
)

Map


In [ ]:
# Convert all predictor bands to Float32
predictors_float = predictors.select([
    "elevation",
    "slope",
    "ndvi",
    "ascending_change",
    "descending_change"
]).toFloat()

# Submit a new export
export_task = ee.batch.Export.image.toDrive(
    image=predictors_float,
    description="FloodSense_XGBoost_Predictors",
    folder="FloodSense_AI",
    fileNamePrefix="FloodSense_predictors",
    region=roi,
    scale=20,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

export_task.start()

print("New export submitted!")
print("Task ID:", export_task.id)


In [ ]:
print(export_task.status())

In [ ]:
import os
import rasterio

tif_path = str(DATA_DIR / "FloodSense_predictors.tif")

print("File exists:", os.path.exists(tif_path))

with rasterio.open(tif_path) as src:
    print("Raster dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Coordinate system:", src.crs)
    print("Band descriptions:", src.descriptions)

In [ ]:
import numpy as np
import rasterio

input_tif = tif_path
output_tif = "/content/FloodSense_XGBoost_Probability.tif"

with rasterio.open(input_tif) as src:

    profile = src.profile.copy()
    profile.update(
        count=1,
        dtype="float32",
        nodata=-9999,
        compress="lzw"
    )

    with rasterio.open(output_tif, "w", **profile) as dst:

        for _, window in src.block_windows(1):

            data = src.read(window=window).astype("float32")

            valid = np.all(src.read_masks(window=window) > 0, axis=0)
            valid &= np.all(np.isfinite(data), axis=0)

            result = np.full(valid.shape, -9999, dtype="float32")

            if np.any(valid):
                pixels = np.moveaxis(data, 0, -1)[valid]
                result[valid] = model.predict_proba(pixels)[:, 1]

            dst.write(result, 1, window=window)

print("XGBoost probability raster created!")
print(output_tif)

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

# Read probability raster
with rasterio.open(output_tif) as src:
    probability = src.read(1)
    bounds = src.bounds
    nodata = src.nodata

# Mask NoData pixels
probability = np.ma.masked_where(
    (probability == nodata) | (~np.isfinite(probability)),
    probability
)

# District boundary
district_gdf = geemap.ee_to_gdf(nowshera)
district_gdf = district_gdf.to_crs("EPSG:4326")

# Create map
fig, ax = plt.subplots(figsize=(11, 10))

image = ax.imshow(
    probability,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin="upper",
    cmap="YlOrRd",
    vmin=0,
    vmax=1
)

district_gdf.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

# Probability legend
cbar = plt.colorbar(image, ax=ax, shrink=0.72, pad=0.02)
cbar.set_label("Predicted flood-reference probability", fontsize=11)

ax.set_title(
    "XGBoost Flood Probability Map\nNowshera District, Pakistan | August 2025",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2, linestyle="--")

# North arrow
ax.annotate(
    "N",
    xy=(0.94, 0.94),
    xytext=(0.94, 0.85),
    xycoords="axes fraction",
    ha="center",
    fontsize=13,
    fontweight="bold",
    arrowprops=dict(facecolor="black", width=2, headwidth=8)
)

plt.tight_layout()

map_path = "/content/FloodSense_XGBoost_Probability_Map.png"
plt.savefig(map_path, dpi=300, bbox_inches="tight")
plt.show()

print("Map saved:", map_path)
pdf_path = "/content/FloodSense_XGBoost_Probability_Map.pdf"

fig.savefig(
    pdf_path,
    format="pdf",
    bbox_inches="tight",
    dpi=300
)

print("Research PDF saved:", pdf_path)

In [ ]:
flood_export = ee.batch.Export.image.toDrive(
    image=final_flood_map,
    description="FloodSense_Sentinel1_Candidate_Map",
    folder="FloodSense_AI",
    fileNamePrefix="FloodSense_Sentinel1_Candidates",
    region=roi,
    scale=20,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

flood_export.start()

print("Export submitted!")
print("Task ID:", flood_export.id)

In [ ]:
import rasterio
import numpy as np
import os

flood_tif = str(DATA_DIR / "FloodSense_Sentinel1_Candidates.tif")

print("File exists:", os.path.exists(flood_tif))

with rasterio.open(flood_tif) as src:
    sentinel_map = src.read(1)
    flood_bounds = src.bounds
    flood_nodata = src.nodata

    print("Raster dimensions:", src.width, "x", src.height)
    print("CRS:", src.crs)
    print("Unique pixel values:", np.unique(sentinel_map))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# Mask zero/background pixels
sentinel_display = np.ma.masked_equal(sentinel_map, 0)

# Three category colours
colors = ["#2C7BB6", "#F4A259", "#2A9D8F"]
cmap = ListedColormap(colors)
cmap.set_bad("white")

fig, ax = plt.subplots(figsize=(11, 10))

ax.imshow(
    sentinel_display,
    extent=[
        flood_bounds.left, flood_bounds.right,
        flood_bounds.bottom, flood_bounds.top
    ],
    origin="upper",
    cmap=cmap,
    norm=BoundaryNorm([0.5, 1.5, 2.5, 3.5], cmap.N)
)

# District boundary
district_gdf.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

# Legend
from matplotlib.patches import Patch

legend_items = [
    Patch(facecolor=colors[0], label="Ascending orbit only"),
    Patch(facecolor=colors[1], label="Descending orbit only"),
    Patch(facecolor=colors[2], label="Both orbits")
]

ax.legend(handles=legend_items, loc="lower left", frameon=True, title="Candidate categories")

ax.set_title(
    "Sentinel-1 SAR Flood-Related Change Detection\nNowshera District, Pakistan | August 2025",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2, linestyle="--")

# North arrow
ax.annotate(
    "N",
    xy=(0.94, 0.94),
    xytext=(0.94, 0.85),
    xycoords="axes fraction",
    ha="center",
    fontsize=13,
    fontweight="bold",
    arrowprops=dict(facecolor="black", width=2, headwidth=8)
)

plt.tight_layout()

# Save PNG and PDF
sentinel_png = "/content/FloodSense_Sentinel1_Candidate_Map.png"
sentinel_pdf = "/content/FloodSense_Sentinel1_Candidate_Map.pdf"

fig.savefig(sentinel_png, dpi=300, bbox_inches="tight")
fig.savefig(sentinel_pdf, format="pdf", dpi=300, bbox_inches="tight")

plt.show()

print("PNG saved:", sentinel_png)
print("PDF saved:", sentinel_pdf)

In [ ]:
# Convert Sentinel-1 candidate map to binary
sentinel_binary = final_flood_map.gt(0).unmask(0)

# Convert UNOSAT reference to binary
unosat_binary = reference_image.gt(0).unmask(0)

# Create comparison categories
comparison_map = (
    ee.Image(0)
    .where(sentinel_binary.eq(1).And(unosat_binary.eq(0)), 1)
    .where(sentinel_binary.eq(0).And(unosat_binary.eq(1)), 2)
    .where(sentinel_binary.eq(1).And(unosat_binary.eq(1)), 3)
    .rename("Comparison")
    .clip(roi)
    .toByte()
)

# Export to Google Drive
comparison_export = ee.batch.Export.image.toDrive(
    image=comparison_map,
    description="FloodSense_UNOSAT_Comparison",
    folder="FloodSense_AI",
    fileNamePrefix="FloodSense_UNOSAT_Comparison",
    region=roi,
    scale=20,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

comparison_export.start()

print("Comparison raster export submitted!")
print("Task ID:", comparison_export.id)

In [ ]:
import rasterio
import numpy as np
import os

comparison_tif = str(DATA_DIR / "FloodSense_UNOSAT_Comparison.tif")

print("File exists:", os.path.exists(comparison_tif))

with rasterio.open(comparison_tif) as src:
    comparison_data = src.read(1)
    comparison_bounds = src.bounds
    comparison_nodata = src.nodata

    print("Dimensions:", src.width, "x", src.height)
    print("CRS:", src.crs)
    print("Unique values:", np.unique(comparison_data))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# Four comparison categories
colors = [
    "#F2F2F2",  # Neither
    "#2878B5",  # Sentinel-1 only
    "#E69F00",  # UNOSAT only
    "#168C70"   # Overlap
]

cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

fig, ax = plt.subplots(figsize=(11, 10))

ax.imshow(
    comparison_data,
    extent=[
        comparison_bounds.left,
        comparison_bounds.right,
        comparison_bounds.bottom,
        comparison_bounds.top
    ],
    origin="upper",
    cmap=cmap,
    norm=norm
)

# District boundary
district_gdf.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

# Legend
legend_items = [
    Patch(facecolor=colors[0], edgecolor="gray", label="Neither dataset"),
    Patch(facecolor=colors[1], label="Sentinel-1 only"),
    Patch(facecolor=colors[2], label="UNOSAT only"),
    Patch(facecolor=colors[3], label="Overlap")
]

ax.legend(
    handles=legend_items,
    loc="lower left",
    title="Spatial comparison",
    frameon=True
)

ax.set_title(
    "Sentinel-1 vs UNOSAT Flood Reference Comparison\nNowshera District, Pakistan | August 2025",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2, linestyle="--")

# North arrow
ax.annotate(
    "N",
    xy=(0.94, 0.94),
    xytext=(0.94, 0.85),
    xycoords="axes fraction",
    ha="center",
    fontsize=13,
    fontweight="bold",
    arrowprops=dict(facecolor="black", width=2, headwidth=8)
)

plt.tight_layout()

# Save PNG and PDF
comparison_png = "/content/FloodSense_UNOSAT_Comparison_Map.png"
comparison_pdf = "/content/FloodSense_UNOSAT_Comparison_Map.pdf"

fig.savefig(comparison_png, dpi=300, bbox_inches="tight")
fig.savefig(comparison_pdf, format="pdf", dpi=300, bbox_inches="tight")

plt.show()

print("PNG saved:", comparison_png)
print("PDF saved:", comparison_pdf)

In [ ]:
print("Nowshera boundary features:", len(district_gdf))
print("Coordinate system:", district_gdf.crs)
print("Bounds:", district_gdf.total_bounds)

district_gdf.plot(
    figsize=(7, 7),
    edgecolor="black",
    facecolor="lightblue"
)

In [ ]:
import requests
import geopandas as gpd

# Get Pakistan's actual national boundary metadata
url = "https://www.geoboundaries.org/api/current/gbOpen/PAK/ADM0/"

response = requests.get(url)
metadata = response.json()

print("Country:", metadata["boundaryName"])
print("Boundary download:", metadata["gjDownloadURL"])

In [ ]:
import requests
import geopandas as gpd

pakistan_url = metadata["gjDownloadURL"]

response = requests.get(pakistan_url, timeout=60)
response.raise_for_status()

with open("/content/Pakistan_ADM0.geojson", "wb") as f:
    f.write(response.content)

pakistan_gdf = gpd.read_file("/content/Pakistan_ADM0.geojson")

pakistan_gdf = pakistan_gdf.to_crs("EPSG:4326")

print("Pakistan boundary loaded successfully")
print("Features:", len(pakistan_gdf))
print("CRS:", pakistan_gdf.crs)
print("Bounds:", pakistan_gdf.total_bounds)

In [ ]:
# Retrieve actual provincial boundaries from geoBoundaries
kp_url = "https://www.geoboundaries.org/api/current/gbOpen/PAK/ADM1/"

kp_response = requests.get(kp_url, timeout=60)
kp_response.raise_for_status()

kp_metadata = kp_response.json()

print("Dataset:", kp_metadata["boundaryName"])
print("Download URL:", kp_metadata["gjDownloadURL"])

In [ ]:
kp_response = requests.get(kp_metadata["gjDownloadURL"], timeout=60)
kp_response.raise_for_status()

with open("/content/Pakistan_ADM1.geojson", "wb") as f:
    f.write(kp_response.content)

provinces_gdf = gpd.read_file("/content/Pakistan_ADM1.geojson")
provinces_gdf = provinces_gdf.to_crs("EPSG:4326")

print("Total administrative units:", len(provinces_gdf))
print("Available columns:", provinces_gdf.columns.tolist())

# Inspect province names
print(provinces_gdf.iloc[:, 0:4].head(10))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(17, 9))

ax1, ax2, ax3 = axes

# (a) PAKISTAN
pakistan_gdf.plot(
    ax=ax1,
    color="#E8E8E8",
    edgecolor="black",
    linewidth=0.8
)

provinces_gdf.boundary.plot(
    ax=ax1,
    color="white",
    linewidth=0.7
)

kp_gdf.plot(
    ax=ax1,
    color="#75AADB",
    edgecolor="black",
    linewidth=1
)

ax1.scatter(
    nowshera_centre.x,
    nowshera_centre.y,
    color="red",
    s=35,
    edgecolor="black",
    zorder=5
)

ax1.set_xlim(60, 78)
ax1.set_ylim(23, 38)
ax1.set_title("(a) Pakistan", fontweight="bold")


# (b) KHYBER PAKHTUNKHWA
kp_gdf.plot(
    ax=ax2,
    color="#E8E8E8",
    edgecolor="black",
    linewidth=1
)

district_gdf.plot(
    ax=ax2,
    color="#E76F51",
    edgecolor="black",
    linewidth=1.2
)

minx, miny, maxx, maxy = kp_gdf.total_bounds
ax2.set_xlim(minx - 0.3, maxx + 0.3)
ax2.set_ylim(miny - 0.3, maxy + 0.3)

ax2.set_title(
    "(b) Khyber Pakhtunkhwa",
    fontweight="bold"
)


# (c) NOWSHERA DISTRICT
district_gdf.plot(
    ax=ax3,
    color="#F4A259",
    edgecolor="black",
    linewidth=1.5
)

ax3.set_title("(c) Nowshera District", fontweight="bold")

# Set a close but balanced extent
minx, miny, maxx, maxy = district_gdf.total_bounds
xpad = (maxx - minx) * 0.12
ypad = (maxy - miny) * 0.12

ax3.set_xlim(minx - xpad, maxx + xpad)
ax3.set_ylim(miny - ypad, maxy + ypad)

# Common map formatting
for ax in axes:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.2, linestyle="--")

    ax.annotate(
        "N",
        xy=(0.94, 0.94),
        xytext=(0.94, 0.85),
        xycoords="axes fraction",
        ha="center",
        fontweight="bold",
        arrowprops=dict(facecolor="black", width=1.5, headwidth=7)
    )

# Main title
fig.suptitle(
    "Geographical Location of the Study Area: Nowshera District, Pakistan",
    fontsize=15,
    fontweight="bold"
)

# Shared legend
legend_items = [
    Patch(facecolor="#75AADB", edgecolor="black",
          label="Khyber Pakhtunkhwa"),
    Patch(facecolor="#E76F51", edgecolor="black",
          label="Nowshera District"),
    Patch(facecolor="#E8E8E8", edgecolor="black",
          label="Other administrative areas")
]

fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, 0.04)
)

fig.text(
    0.5, 0.01,
    "Sources: geoBoundaries PAK ADM0 and ADM1; Nowshera study-area boundary.",
    ha="center",
    fontsize=8
)

plt.tight_layout(rect=[0, 0.11, 1, 0.92])

plt.savefig(
    "/content/FloodSense_Study_Area_Map.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "/content/FloodSense_Study_Area_Map.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()

print("Three-panel study-area map exported as PNG and PDF.")

In [ ]:
import shap
import matplotlib.pyplot as plt

# Select 300 test samples
X_explain = X_test.sample(
    n=min(300, len(X_test)),
    random_state=42
)

# Explain the trained XGBoost model
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_explain)

# Generate SHAP summary plot
shap.summary_plot(
    shap_values,
    X_explain,
    plot_type="dot",
    show=False
)

plt.title(
    "SHAP Summary: XGBoost Flood Prediction Model",
    fontsize=13,
    fontweight="bold",
    pad=15
)

plt.tight_layout()

# Export PNG and PDF
plt.savefig(
    "/content/FloodSense_SHAP_Explainability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "/content/FloodSense_SHAP_Explainability.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()

print("SHAP figure saved in PNG and PDF.")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Open the XGBoost probability raster
probability_path = "/content/FloodSense_XGBoost_Probability.tif"

with rasterio.open(probability_path) as src:
    probability = src.read(1)
    bounds = src.bounds
    nodata = src.nodata

# Mask invalid pixels
valid = np.isfinite(probability)

if nodata is not None:
    valid &= probability != nodata

# Apply the 0.50 classification threshold
classified = np.where(probability >= 0.50, 1, 0)
classified = np.ma.masked_where(~valid, classified)

# Plot the classified map
fig, ax = plt.subplots(figsize=(10, 9))

ax.imshow(
    classified,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin="upper",
    cmap="RdYlBu_r",
    vmin=0,
    vmax=1,
    interpolation="nearest"
)

# Add Nowshera district boundary
district_gdf.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

legend_items = [
    Patch(facecolor="#313695", label="Not predicted as flood-reference class"),
    Patch(facecolor="#a50026", label="Predicted as flood-reference class")
]

ax.legend(handles=legend_items, loc="lower right", fontsize=8)

ax.set_title(
    "XGBoost Predicted Class Map – Nowshera District, Pakistan",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    "/content/FloodSense_XGBoost_Classified_Map.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "/content/FloodSense_XGBoost_Classified_Map.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import os
import shutil

# Create the final project output folder
output_folder = str(DATA_DIR / "Final_Outputs")
os.makedirs(output_folder, exist_ok=True)

# Files to collect
files = [
    "FloodSense_Study_Area_Map.png",
    "FloodSense_Study_Area_Map.pdf",
    "FloodSense_Sentinel1_Candidate_Map.png",
    "FloodSense_Sentinel1_Candidate_Map.pdf",
    "FloodSense_UNOSAT_Comparison_Map.png",
    "FloodSense_UNOSAT_Comparison_Map.pdf",
    "FloodSense_XGBoost_Probability_Map.png",
    "FloodSense_XGBoost_Probability_Map.pdf",
    "FloodSense_XGBoost_Classified_Map.png",
    "FloodSense_XGBoost_Classified_Map.pdf",
    "FloodSense_SHAP_Explainability.png",
    "FloodSense_SHAP_Explainability.pdf"
]

# Copy existing files
for filename in files:
    source = "/content/" + filename

    if os.path.exists(source):
        shutil.copy2(source, output_folder)
        print("COPIED:", filename)
    else:
        print("NOT FOUND:", filename)

print("\nFinal output folder:", output_folder)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Overall model metrics
metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1_Score": f1_score(y_test, y_pred),
    "ROC_AUC": roc_auc_score(y_test, y_probability)
}])

metrics.to_csv(
    "/content/FloodSense_XGBoost_Model_Metrics.csv",
    index=False
)

# Detailed classification report
report = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        output_dict=True
    )
).transpose()

report.to_csv(
    "/content/FloodSense_Classification_Report.csv"
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_Not_Reference", "Actual_Reference"],
    columns=["Predicted_Not_Reference", "Predicted_Reference"]
)

cm_df.to_csv(
    "/content/FloodSense_Confusion_Matrix.csv"
)

print("Three statistical CSV files created successfully.")
display(metrics)
display(cm_df)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

model_metrics = pd.DataFrame([{
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1_Score": f1_score(y_test, y_pred),
    "ROC_AUC": roc_auc_score(y_test, y_probability),
    "Test_Samples": len(y_test)
}])

model_metrics.to_csv(
    "/content/FloodSense_XGBoost_Model_Metrics.csv",
    index=False
)

display(model_metrics)

print("Model metrics CSV saved successfully.")

In [ ]:
import pandas as pd

flood_statistics = pd.DataFrame({
    "Category": [
        "Ascending orbit only",
        "Descending orbit only",
        "Detected by both orbits",
        "Combined candidate area"
    ],
    "Area_km2": [
        3.69,
        5.60,
        2.67,
        11.96
    ]
})

flood_statistics.to_csv(
    "/content/FloodSense_Sentinel1_Area_Statistics.csv",
    index=False
)

display(flood_statistics)

print("Sentinel-1 area statistics CSV created.")

In [ ]:
import numpy as np
import pandas as pd

# Pixel area in km² from the comparison raster
with rasterio.open(
    str(DATA_DIR / "FloodSense_UNOSAT_Comparison.tif")
) as src:
    comparison = src.read(1)
    transform = src.transform
    pixel_area_km2 = abs(transform.a * transform.e) * 111.32 * 111.32

# Classes:
# 0 = Neither
# 1 = Sentinel-1 only
# 2 = UNOSAT only
# 3 = Overlap

valid = np.isin(comparison, [0, 1, 2, 3])

sentinel_area = np.sum(np.isin(comparison[valid], [1, 3])) * pixel_area_km2
unosat_area = np.sum(np.isin(comparison[valid], [2, 3])) * pixel_area_km2
overlap_area = np.sum(comparison[valid] == 3) * pixel_area_km2
union_area = np.sum(np.isin(comparison[valid], [1, 2, 3])) * pixel_area_km2

iou = overlap_area / union_area if union_area else 0

comparison_stats = pd.DataFrame([{
    "Sentinel1_Candidate_km2": sentinel_area,
    "UNOSAT_Reference_km2": unosat_area,
    "Overlap_km2": overlap_area,
    "Union_km2": union_area,
    "IoU": iou
}])

comparison_stats.to_csv(
    "/content/FloodSense_UNOSAT_Comparison_Statistics.csv",
    index=False
)

display(comparison_stats)

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files

shap_importance = pd.DataFrame({
    "Feature": X_explain.columns,
    "Mean_Absolute_SHAP": np.abs(shap_values).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

shap_importance.to_csv(
    "/content/FloodSense_SHAP_Feature_Importance.csv",
    index=False
)

display(shap_importance)

files.download(
    "/content/FloodSense_SHAP_Feature_Importance.csv"
)

In [ ]:
import rasterio
import numpy as np

files_to_check = {
    "Sentinel-1 Candidate":
    str(DATA_DIR / "FloodSense_Sentinel1_Candidates.tif"),

    "UNOSAT Comparison":
    str(DATA_DIR / "FloodSense_UNOSAT_Comparison.tif")
}

for name, path in files_to_check.items():

    with rasterio.open(path) as src:

        data = src.read(1)

        print("\n", name)
        print("-----------------------------")
        print("CRS:", src.crs)
        print("Dimensions:", src.width, "x", src.height)
        print("Pixel size:", src.res)
        print("NoData:", src.nodata)
        print("Pixel values:", np.unique(data, return_counts=True))

In [ ]:
import rasterio
import numpy as np
from pyproj import Geod

geod = Geod(ellps="WGS84")

raster_paths = {
    "Sentinel-1 Candidate":
    str(DATA_DIR / "FloodSense_Sentinel1_Candidates.tif"),

    "UNOSAT Comparison":
    str(DATA_DIR / "FloodSense_UNOSAT_Comparison.tif")
}

for name, path in raster_paths.items():

    total_area_m2 = 0

    with rasterio.open(path) as src:

        for row in range(src.height):

            data = src.read(1, window=((row, row+1), (0, src.width)))[0]

            if name == "Sentinel-1 Candidate":
                candidate_pixels = np.isin(data, [1, 2, 3])
            else:
                candidate_pixels = np.isin(data, [1, 3])

            pixel_count = np.count_nonzero(candidate_pixels)

            if pixel_count == 0:
                continue

            x0, y0 = src.transform * (0, row)
            x1, y1 = src.transform * (1, row + 1)

            area, _ = geod.polygon_area_perimeter(
                [x0, x1, x1, x0],
                [y0, y0, y1, y1]
            )

            total_area_m2 += abs(area) * pixel_count

    print(name, "=", round(total_area_m2 / 1_000_000, 4), "km²")

In [ ]:
import rasterio
import numpy as np
import pandas as pd
from pyproj import Geod

geod = Geod(ellps="WGS84")

path = str(DATA_DIR / "FloodSense_UNOSAT_Comparison.tif")

sentinel_area = 0
unosat_area = 0
overlap_area = 0
union_area = 0

with rasterio.open(path) as src:

    for row in range(src.height):

        data = src.read(
            1,
            window=((row, row + 1), (0, src.width))
        )[0]

        x0, y0 = src.transform * (0, row)
        x1, y1 = src.transform * (1, row + 1)

        pixel_area, _ = geod.polygon_area_perimeter(
            [x0, x1, x1, x0],
            [y0, y0, y1, y1]
        )

        pixel_area = abs(pixel_area)

        sentinel_area += np.count_nonzero(
            np.isin(data, [1, 3])
        ) * pixel_area

        unosat_area += np.count_nonzero(
            np.isin(data, [2, 3])
        ) * pixel_area

        overlap_area += np.count_nonzero(data == 3) * pixel_area

        union_area += np.count_nonzero(
            np.isin(data, [1, 2, 3])
        ) * pixel_area

iou = overlap_area / union_area

corrected_stats = pd.DataFrame([{
    "Sentinel1_Candidate_km2": sentinel_area / 1e6,
    "UNOSAT_Reference_km2": unosat_area / 1e6,
    "Overlap_km2": overlap_area / 1e6,
    "Union_km2": union_area / 1e6,
    "IoU": iou
}])

corrected_stats.to_csv(
    "/content/FloodSense_UNOSAT_Comparison_Statistics.csv",
    index=False
)

display(corrected_stats)